In [0]:
dbutils.widgets.text("catalog", "oceanwatch_e2")
dbutils.widgets.text("fecha", "2023-06-07")
dbutils.widgets.text("max_cuarentena_pct", "1.0")
catalog = dbutils.widgets.get("catalog")
fecha = dbutils.widgets.get("fecha")
max_pct = float(dbutils.widgets.get("max_cuarentena_pct"))
print("catalog:", catalog, "| fecha:", fecha, "| umbral cuarentena:", max_pct, "%")

In [0]:
from pyspark.sql import functions as F

tbl_metr = f"{catalog}.ops.calidad_metricas"
tbl_silver = f"{catalog}.silver.ais_posiciones"
tbl_log = f"{catalog}.ops.quality_gate_log"

m = {r.metrica: r.v for r in spark.table(tbl_metr).filter(f"fecha = DATE'{fecha}'")
                                   .groupBy("metrica").agg(F.sum("valor").alias("v")).collect()}
entrada = int(m.get("filas_entrada", 0))
cuarentena = int(m.get("cuarentena", 0))
pct = round(100 * cuarentena / entrada, 4) if entrada else None
plata = spark.table(tbl_silver).filter(f"fecha = DATE'{fecha}'").count()

problemas = []
if entrada == 0:
    problemas.append("sin filas de entrada para la fecha")
if plata == 0:
    problemas.append("plata vacía para la fecha")
if pct is not None and pct > max_pct:
    problemas.append(f"cuarentena {pct}% > umbral {max_pct}%")
resultado = "OK" if not problemas else "FALLA: " + "; ".join(problemas)

(spark.createDataFrame([(fecha, entrada, cuarentena, pct, plata, max_pct, resultado)],
                       "fecha string, filas_entrada long, cuarentena long, cuarentena_pct double, "
                       "filas_plata long, umbral_pct double, resultado string")
      .withColumn("fecha", F.to_date("fecha"))
      .withColumn("ts_evaluacion", F.current_timestamp())
      .write.mode("append").saveAsTable(tbl_log))

print(f"fecha {fecha} | entrada {entrada:,} | cuarentena {cuarentena:,} ({pct}%) | plata {plata:,} → {resultado}")
if problemas:
    raise Exception(f"Quality gate no superado para {fecha}: {resultado}")